In [1]:
from toyaikit.mcp import MCPClient, SubprocessMCPTransport

command = "uv run python main.py".split()
workdir = "faq-mcp"

client = MCPClient(
    transport=SubprocessMCPTransport(
        server_command=command,
        workdir=workdir
    )
)

In [2]:
client.start_server()

client.initialize()
client.initialized()

client.get_tools()

Started server with command: uv run python main.py
Sending initialize request...
Initialize response: {'protocolVersion': '2024-11-05', 'capabilities': {'logging': {}, 'prompts': {'listChanged': True}, 'resources': {'subscribe': False, 'listChanged': True}, 'tools': {'listChanged': True}}, 'serverInfo': {'name': 'Demo 🚀', 'version': '4.1.0'}}
Sending initialized notification...
Handshake completed successfully
Retrieving available tools...
Available tools: ['add_entry', 'search']


[{'_meta': {'fastmcp': {'tags': []}},
  'description': 'Add a new entry to the FAQ database.',
  'inputSchema': {'properties': {'question': {'type': 'string',
     'description': 'The question to be added to the FAQ database.'},
    'answer': {'type': 'string',
     'description': 'The corresponding answer to the question.'}},
   'required': ['question', 'answer'],
   'type': 'object',
   'additionalProperties': False},
  'name': 'add_entry',
  'title': 'Add Entry'},
 {'_meta': {'fastmcp': {'tags': []}},
  'description': 'Search the FAQ database for entries matching the given query.',
  'inputSchema': {'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course FAQ.'}},
   'required': ['query'],
   'type': 'object',
   'additionalProperties': False},
  'name': 'search',
  'outputSchema': {'properties': {'result': {'items': {'additionalProperties': True,
      'type': 'object'},
     'type': 'array'}},
   'required': ['result'],
   'type': 

In [3]:
result = client.call_tool(
    'search',
    {'query': 'how do I run docker?'}
)

result

Calling tool 'search' with arguments: {'query': 'how do I run docker?'}


{'_meta': {'fastmcp': {'wrap_result': True}},
 'content': [{'text': '[{"text":"You may have this error:\\n$ docker run -it ubuntu bash\\nthe input device is not a TTY. If you are using mintty, try prefixing the command with \'winpty\'\\nerror:\\nSolution:\\nUse winpty before docker command (source)\\n$ winpty docker run -it ubuntu bash\\nYou also can make an alias:\\necho \\"alias docker=\'winpty docker\'\\" >> ~/.bashrc\\nOR\\necho \\"alias docker=\'winpty docker\'\\" >> ~/.bash_profile","section":"Module 1: Docker and Terraform","question":"Docker - The input device is not a TTY (Docker run for Windows)","course":"data-engineering-zoomcamp"},{"text":"Make sure you\'re able to start the Docker daemon, and check the issue immediately down below:\\nAnd don’t forget to update the wsl in powershell the  command is wsl –update","section":"Module 1: Docker and Terraform","question":"Docker - Cannot connect to Docker daemon at unix:///var/run/docker.sock. Is the docker daemon running?","cour

In [4]:
from toyaikit.mcp import MCPTools

mcp_tools = MCPTools(client)

In [5]:
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner

In [8]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

For course-related questions, always search the FAQ database first using the available MCP tools.

Use the search results as the basis for your answer.
Do not answer from general knowledge before searching.
""".strip()

chat_interface = IPythonChatInterface()

runner = OpenAIResponsesRunner(
    tools=mcp_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-4o-mini")
)

In [9]:
runner.run()

-> Response received
Calling tool 'search' with arguments: {'query': 'run Docker'}


-> Response received


Chat ended.


LoopResult(new_messages=[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nFor course-related questions, always search the FAQ database first using the available MCP tools.\n\nUse the search results as the basis for your answer.\nDo not answer from general knowledge before searching.", role='developer', phase=None, type=None), EasyInputMessage(content='How do I run Docker?', role='user', phase=None, type=None), ResponseFunctionToolCall(arguments='{"query":"run Docker"}', call_id='call_l7bFpFXeEJapR4FHmPECnmKd', name='search', type='function_call', id='fc_0806b1f931a66921006acc81d9971887d1820c2a97690ef778', async_=None, caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_l7bFpFXeEJapR4FHmPECnmKd', 'output': '[{"text":"You may have this error:\\n$ docker run -it ubuntu bash\\nthe input device is not a TTY. If you are using mintty, try prefixing the 